In [ ]:
# Cell 1 - run information (required)
import os, subprocess, sys, random
import numpy as np, torch
if os.path.isdir("/content"):                       # Colab: upload repo.bundle (git bundle create repo.bundle zoheb)
    os.chdir("/content")
    if not os.path.isdir("data266-lab2-team32"):
        from google.colab import files
        files.upload()
        subprocess.run(["git", "clone", "-q", "-b", "zoheb", "repo.bundle", "data266-lab2-team32"], check=True)
    os.chdir("data266-lab2-team32")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

# Part 2 - Baseline (Zoheb): small end-to-end pipeline
DATA 266 Lab 2, Pair_Programming_Team_32. The brief requires the partner who does not lead Part 2 to implement and run
the complete pipeline, at least as a working baseline. This notebook is that baseline, written separately from the
main pipeline in `part2_diffusion/src/` (Shreya, lead): data preparation, one small LoRA run, generation for base vs
LoRA, and CLIP-score evaluation. Same base model, dataset, seed and split rule as the main pipeline; a smaller budget
(500 steps instead of 1500) and 5 of the 20 fixed prompts. Runs on one Colab T4 in about 30 minutes.

In [ ]:
# packages (diffusers and peft are not on Colab; torchao breaks peft's LoRA loading on Colab)
if os.path.isdir("/content"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.42.1", "peft==0.21.2"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"])
from importlib.metadata import version
print({p: version(p) for p in ["torch", "diffusers", "transformers", "peft", "datasets"]})
BASE, DATASET, CLIP = "stable-diffusion-v1-5/stable-diffusion-v1-5", "lambdalabs/naruto-blip-captions", "openai/clip-vit-base-patch32"
RANK, STEPS, BATCH, LR, RES = 4, 500, 4, 1e-4, 512

## 1. Data preparation
Remove images that cannot be decoded and exact duplicates (same MD5 of the file bytes), split 90/10 with seed 266,
resize to 512x512. Test images are not used for training.

In [ ]:
import hashlib, io
from datasets import load_dataset, Image as HFImage
from PIL import Image

raw = load_dataset(DATASET, split="train").cast_column("image", HFImage(decode=False))
seen, rows, removed = set(), [], {"corrupted": 0, "duplicate": 0}
for r in raw:
    b = r["image"]["bytes"]
    try:
        img = Image.open(io.BytesIO(b)).convert("RGB")
    except Exception:
        removed["corrupted"] += 1
        continue
    h = hashlib.md5(b).hexdigest()
    if h in seen:
        removed["duplicate"] += 1
        continue
    seen.add(h)
    rows.append((img.resize((RES, RES), Image.BICUBIC), r["text"]))

order = np.random.default_rng(SEED).permutation(len(rows))
n_test = round(0.1 * len(rows))
test, train = [rows[i] for i in order[:n_test]], [rows[i] for i in order[n_test:]]
prep = {"loaded": len(raw), "removed": removed, "kept": len(rows), "train": len(train), "test": len(test)}
print(prep)
print("example captions:", [c for _, c in train[:3]])

## 2. Fine-tuning: one LoRA run (rank 4)
LoRA on the UNet attention projections (`to_q`, `to_k`, `to_v`, `to_out.0`); text encoder and VAE frozen in fp16.
Standard noise-prediction loss with the DDPM scheduler, fp16 autocast and gradient checkpointing for the T4.

In [ ]:
import time
import torch.nn.functional as F
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from peft import LoraConfig
from transformers import CLIPTextModel, CLIPTokenizer

dev = "cuda"
tok = CLIPTokenizer.from_pretrained(BASE, subfolder="tokenizer")
text_enc = CLIPTextModel.from_pretrained(BASE, subfolder="text_encoder", torch_dtype=torch.float16).to(dev)
vae = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=torch.float16).to(dev)
unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet").to(dev)
sched = DDPMScheduler.from_pretrained(BASE, subfolder="scheduler")
for m in (text_enc, vae, unet):
    m.requires_grad_(False)
unet.add_adapter(LoraConfig(r=RANK, lora_alpha=RANK, init_lora_weights="gaussian",
                            target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
unet.enable_gradient_checkpointing()
params = [p for p in unet.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR)
scaler = torch.amp.GradScaler("cuda")

def to_tensor(img):
    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 127.5 - 1).permute(2, 0, 1)

gen = torch.Generator().manual_seed(SEED)
torch.cuda.reset_peak_memory_stats()
t0, losses = time.time(), []
unet.train()
for step in range(STEPS):
    idx = torch.randint(len(train), (BATCH,), generator=gen).tolist()
    pix = torch.stack([to_tensor(train[i][0]) for i in idx]).to(dev, torch.float16)
    ids = tok([train[i][1] for i in idx], padding="max_length", max_length=tok.model_max_length,
              truncation=True, return_tensors="pt").input_ids.to(dev)
    with torch.no_grad():
        latents = vae.encode(pix).latent_dist.sample() * vae.config.scaling_factor
        cond = text_enc(ids)[0]
    noise = torch.randn_like(latents)
    t = torch.randint(0, sched.config.num_train_timesteps, (BATCH,), device=dev)
    with torch.autocast("cuda", dtype=torch.float16):
        pred = unet(sched.add_noise(latents, noise, t), t, cond).sample
    loss = F.mse_loss(pred.float(), noise.float())
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    losses.append(loss.item())
    if step % 100 == 0 or step == STEPS - 1:
        print(f"step {step:4d}  loss {np.mean(losses[-100:]):.4f}")

train_stats = {"rank": RANK, "trainable_params": sum(p.numel() for p in params), "learning_rate": LR,
               "steps": STEPS, "batch_size": BATCH, "peak_gpu_memory_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
               "train_minutes": round((time.time() - t0) / 60, 1)}
train_stats["gpu_hours"] = round(train_stats["train_minutes"] / 60, 3)
print(train_stats)

In [ ]:
# save the adapter, free the training models
from diffusers import StableDiffusionPipeline
from diffusers.utils import convert_state_dict_to_diffusers
from peft.utils import get_peft_model_state_dict
OUT = "/content/p2_baseline_zoheb" if os.path.isdir("/content") else "part2_diffusion/outputs/baseline_zoheb"
os.makedirs(OUT, exist_ok=True)
StableDiffusionPipeline.save_lora_weights(f"{OUT}/lora_r4", unet_lora_layers=convert_state_dict_to_diffusers(
    get_peft_model_state_dict(unet)), safe_serialization=True)
del unet, vae, text_enc, opt
torch.cuda.empty_cache()
print(os.listdir(f"{OUT}/lora_r4"))

## 3. Generation: base vs LoRA
5 of the team's 20 fixed prompts (`part2_diffusion/eval/prompts.csv`: 2 in-domain, 2 out-of-domain, 1 compositional),
seeds 0 and 1, the same sampler as the main pipeline (DPM-Solver++, 30 steps, guidance 7.5). The default NSFW safety
checker stays on.

In [ ]:
import pandas as pd
from diffusers import DPMSolverMultistepScheduler

prompts = pd.read_csv("part2_diffusion/eval/prompts.csv").set_index("prompt_id").loc[["ID01", "ID02", "OD01", "OD02", "CP01"]]
pipe = StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16).to(dev)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

def generate(model):
    out = []
    for pid, r in prompts.iterrows():
        for seed in (0, 1):
            res = pipe(r.prompt, num_inference_steps=30, guidance_scale=7.5,
                       generator=torch.Generator(dev).manual_seed(seed))
            out.append({"model": model, "prompt_id": pid, "category": r.category, "prompt": r.prompt, "seed": seed,
                        "nsfw_flagged": bool(res.nsfw_content_detected and res.nsfw_content_detected[0]),
                        "image": res.images[0]})
    return out

t0 = time.time()
gens = generate("base")
pipe.load_lora_weights(f"{OUT}/lora_r4")
gens += generate("lora_r4")
print(len(gens), "images,", sum(g["nsfw_flagged"] for g in gens), "flagged by the NSFW filter,",
      round((time.time() - t0) / len(gens), 1), "s/image")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(prompts), 4, figsize=(12, 3.2 * len(prompts)))
for row, pid in enumerate(prompts.index):
    for col, (model, seed) in enumerate([("base", 0), ("base", 1), ("lora_r4", 0), ("lora_r4", 1)]):
        g = next(g for g in gens if g["prompt_id"] == pid and g["model"] == model and g["seed"] == seed)
        axes[row, col].imshow(g["image"]); axes[row, col].axis("off")
        axes[row, col].set_title(f"{pid} {model} seed {seed}", fontsize=9)
    axes[row, 0].text(-20, 256, prompts.loc[pid, "prompt"][:40], rotation=90, va="center", ha="right", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/grid.jpg", dpi=80); plt.show()

## 4. Evaluation: CLIP score
CLIP score = 100 x cosine similarity between the prompt and the image (CLIP ViT-B/32), averaged per model and per
prompt category. Images blacked out by the NSFW filter are left out.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
del pipe; torch.cuda.empty_cache()
clip_model, clip_proc = CLIPModel.from_pretrained(CLIP).to(dev).eval(), CLIPProcessor.from_pretrained(CLIP)

@torch.no_grad()
def clip_score(img, text):
    x = clip_proc(text=[text], images=img, return_tensors="pt", padding=True, truncation=True).to(dev)
    i = clip_model.get_image_features(pixel_values=x.pixel_values)
    t = clip_model.get_text_features(input_ids=x.input_ids, attention_mask=x.attention_mask)
    i, t = (getattr(e, "pooler_output", e) for e in (i, t))   # transformers 5 returns an output object, 4.x a tensor
    return max(100 * F.cosine_similarity(i, t).item(), 0)

df = pd.DataFrame([{k: v for k, v in g.items() if k != "image"} | {"clip_score": clip_score(g["image"], g["prompt"])}
                   for g in gens])
scored = df[~df.nsfw_flagged]
table = scored.pivot_table(index="category", columns="model", values="clip_score", aggfunc="mean").round(2)
table.loc["all"] = scored.groupby("model").clip_score.mean().round(2)
print(table)

## Results and save

In [ ]:
import json
results = {"data_prep": prep, "training": train_stats,
           "clip_score": table.to_dict(), "nsfw_flagged": int(df.nsfw_flagged.sum()), "images": len(df)}
print(json.dumps(results, indent=2))
dest = "part2_diffusion/metrics/baseline_zoheb"
os.makedirs(dest, exist_ok=True)
json.dump(results, open(f"{dest}/results.json", "w"), indent=2)
df.to_csv(f"{dest}/clip_scores.csv", index=False)
import shutil
shutil.copy(f"{OUT}/grid.jpg", f"{dest}/grid.jpg")
if os.path.isdir("/content"):     # download metrics + grid (copy into the repo) and the LoRA weights
    shutil.make_archive("/content/p2_baseline_zoheb", "zip", ".", dest)
    shutil.make_archive("/content/p2_baseline_zoheb_lora", "zip", OUT, "lora_r4")
    from google.colab import files
    files.download("/content/p2_baseline_zoheb.zip"); files.download("/content/p2_baseline_zoheb_lora.zip")